# Confidence Analysis
**Does model certainty predict outcome quality?**

Two complementary analyses:
- **Part A (no data needed):** cross-window proxy — do windows where the model is more decisive perform better?
- **Part B (requires DB):** per-prediction — calibration curves, confidence-filtered accuracy, coverage-precision tradeoff on held-out data.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from config import ARTIFACTS_PATH

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Load new-format artifacts
artifacts = {}
for f in sorted(ARTIFACTS_PATH.glob('*.pkl')):
    a = joblib.load(f)
    if 'cv_summary' not in a:
        continue
    artifacts[f.stem] = a
    tt = a.get("target_type", "discrete")
    print(f'  {f.stem:22s}  h={a["horizon"]}  mode={a["mode"]}  '
          f'target={tt}  windows={len(a["cv_metrics"])}')
print(f'\n{len(artifacts)} artifact(s) loaded.')


In [ ]:
# Split by target type
artifacts_discrete = {k: v for k, v in artifacts.items()
                      if v.get("target_type", "discrete") == "discrete"}
artifacts_cont     = {k: v for k, v in artifacts.items()
                      if v.get("target_type", "continuous") == "continuous"}
print(f"Discrete: {len(artifacts_discrete)}  |  Continuous: {len(artifacts_cont)}")


---
## Part A — Cross-Window Confidence Proxy
Each CV window has a `mean_predicted_prob` (average model output) and `pred_positive_rate` (fraction predicted up).
We use **decisiveness** = |mean_predicted_prob − 0.5| as a proxy: a window where the model
averages 0.62 is more confident than one averaging 0.51.

In [ ]:
window_rows = []
for stem, a in artifacts_discrete.items():
    for wi, m in enumerate(a['cv_metrics']):
        window_rows.append({
            'name':               stem,
            'model':              a['model_key'],
            'window':             wi,
            'balanced_accuracy':  m['balanced_accuracy'],
            'roc_auc':            m['roc_auc'],
            'mcc':                m['mcc'],
            'mean_predicted_prob': m.get('mean_predicted_prob', np.nan),
            'pred_positive_rate': m.get('pred_positive_rate', np.nan),
        })

wdf = pd.DataFrame(window_rows)
wdf['decisiveness'] = (wdf['mean_predicted_prob'] - 0.5).abs()
print(wdf.head())

In [ ]:
# Scatter: decisiveness vs balanced_accuracy, one subplot per model
models = wdf['name'].unique()
ncols = min(3, len(models))
nrows = max(1, (len(models) + ncols - 1) // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
palette = sns.color_palette('tab10', len(models))

for ax, (name, color) in zip(axes.flat, zip(models, palette)):
    sub = wdf[wdf['name'] == name].dropna(subset=['decisiveness', 'balanced_accuracy'])
    if sub.empty:
        ax.set_visible(False)
        continue
    ax.scatter(sub['decisiveness'], sub['balanced_accuracy'],
               color=color, alpha=0.75, s=60, edgecolors='white', lw=0.5)
    # regression line
    if len(sub) > 2:
        m_, b_ = np.polyfit(sub['decisiveness'], sub['balanced_accuracy'], 1)
        x_ = np.linspace(sub['decisiveness'].min(), sub['decisiveness'].max(), 50)
        ax.plot(x_, m_ * x_ + b_, color='black', lw=1.2, ls='--')
        r, p = stats.pearsonr(sub['decisiveness'], sub['balanced_accuracy'])
        ax.set_title(f'{name}\nr={r:.2f}, p={p:.3f}', fontsize=9)
    else:
        ax.set_title(name, fontsize=9)
    ax.axhline(0.5, color='red', lw=0.7, ls=':')
    ax.set_xlabel('Decisiveness |mean_prob − 0.5|', fontsize=8)
    ax.set_ylabel('Balanced accuracy', fontsize=8)

for ax in axes.flat[len(models):]:
    ax.set_visible(False)

plt.suptitle('Does higher model confidence → better window performance?', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation summary table
corr_rows = []
for name, sub in wdf.groupby('name'):
    sub = sub.dropna(subset=['decisiveness', 'balanced_accuracy'])
    if len(sub) < 3:
        continue
    r_ba,  p_ba  = stats.pearsonr(sub['decisiveness'], sub['balanced_accuracy'])
    r_auc, p_auc = stats.pearsonr(sub['decisiveness'], sub['roc_auc'])
    r_mcc, p_mcc = stats.pearsonr(sub['decisiveness'], sub['mcc'])
    corr_rows.append({
        'artifact': name,
        'model':    sub['model'].iloc[0],
        'n_windows': len(sub),
        'r(decisiveness, bal_acc)': f'{r_ba:.3f} (p={p_ba:.3f})',
        'r(decisiveness, auc)':    f'{r_auc:.3f} (p={p_auc:.3f})',
        'r(decisiveness, mcc)':    f'{r_mcc:.3f} (p={p_mcc:.3f})',
    })

if corr_rows:
    display(pd.DataFrame(corr_rows).set_index('artifact'))
else:
    print('Not enough windows for correlation analysis.')

---
## Part B — Per-Prediction Confidence Analysis
Loads each final model, runs inference on held-out data (dates after `train_end`),
then analyses probability distributions, calibration, and confidence-filtered performance.

> **Requires SQLite DB access.**  Run the cell below once to collect predictions.

In [ ]:
from db.base import sqlite_connection
from db.sqlite.queries_ohlcv import fetch_ohlcv
from db.utils_ohlcv import get_ibex_tickers, get_macro_tickers
from models.trees.features import ml_ready

def _load_ohlcv():
    ibex_tickers  = get_ibex_tickers()
    macro_tickers = get_macro_tickers()
    with sqlite_connection() as conn:
        df_micro = fetch_ohlcv(ibex_tickers)
        df_macro = fetch_ohlcv(macro_tickers)
    df_micro = df_micro[df_micro['volume'] > 0].reset_index(drop=True)
    df_macro = df_macro.reset_index(drop=True)
    return df_micro, df_macro

def run_inference(artifact: dict, df_micro_raw, df_macro_raw) -> pd.DataFrame:
    """Run final model on data after train_end.  Returns (date, ticker, actual, proba, pred)."""
    import torch

    key        = artifact['model_key']
    horizon    = artifact['horizon']
    ft_type    = artifact['ft_type']
    train_end  = pd.Timestamp(artifact['train_end'])

    df_macro_arg = df_macro_raw if ft_type == 'macro' else None
    df, X, y, mask = ml_ready(horizon, df_micro_raw,
                               df_macro=df_macro_arg, ft_type=ft_type)

    dates   = df.loc[mask, 'date'].reset_index(drop=True)
    tickers = df.loc[mask, 'ticker'].reset_index(drop=True)
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    eval_mask = pd.to_datetime(dates) > train_end
    if eval_mask.sum() == 0:
        print(f'  {key}: no held-out rows after {train_end.date()}')
        return pd.DataFrame()

    if key in ('rf', 'xgb'):
        X_ev = X.loc[eval_mask]
        probas = artifact['model'].predict_proba(X_ev)[:, 1]
        preds  = (probas >= 0.5).astype(int)
        return pd.DataFrame({
            'date':   dates.loc[eval_mask].values,
            'ticker': tickers.loc[eval_mask].values,
            'actual': y.loc[eval_mask].values,
            'proba':  probas,
            'pred':   preds,
        })

    # --- neural ---
    from models.neural.lstm import (add_cyclic_dow, build_sequences,
                                    SequenceDataset, SEQ_LEN)
    if 'dow' in X.columns:
        X = add_cyclic_dow(X.copy())

    seqs, labs, last_dates = build_sequences(X, y, tickers, dates, SEQ_LEN)
    eval_date_set = set(dates.loc[eval_mask].values)
    seq_mask = np.array([d in eval_date_set for d in last_dates])

    seqs_ev = seqs[seq_mask];  labs_ev = labs[seq_mask]
    last_ev = last_dates[seq_mask]
    if len(seqs_ev) == 0:
        print(f'  {key}: no eval sequences')
        return pd.DataFrame()

    sc  = artifact['scaler']
    n, T, f = seqs_ev.shape
    seqs_s  = sc.transform(seqs_ev.reshape(-1, f)).reshape(n, T, f)

    cfg = artifact['model_config']
    if 'num_filters' in cfg:
        from models.neural.cnn_rnn import StockCNNRNN
        model = StockCNNRNN(**cfg)
    else:
        from models.neural.lstm import StockRNN
        model = StockRNN(**cfg)
    model.load_state_dict(artifact['model_state'])
    model.eval()

    with torch.no_grad():
        logits = model(torch.tensor(seqs_s, dtype=torch.float32))
    probas = torch.sigmoid(logits).numpy()
    preds  = (probas >= 0.5).astype(int)

    return pd.DataFrame({
        'date':   last_ev,
        'actual': labs_ev.astype(int),
        'proba':  probas,
        'pred':   preds,
    })

# --- collect ---
from config import load_env
load_env()
df_micro_raw, df_macro_raw = _load_ohlcv()

pred_data = {}   # name -> DataFrame
for name, a in artifacts_discrete.items():
    print(f'Running inference: {name} ...')
    df_pred = run_inference(a, df_micro_raw, df_macro_raw)
    if len(df_pred) > 0:
        pred_data[name] = df_pred
        print(f'  -> {len(df_pred)} predictions, date range: '
              f'{df_pred["date"].min()} -- {df_pred["date"].max()}')

print(f'\nPredictions collected for: {list(pred_data.keys())}')

### B1  Probability Distributions

In [ ]:
if not pred_data:
    print('No prediction data available — run inference cell above.')
else:
    ncols = min(3, len(pred_data))
    nrows = max(1, (len(pred_data) + ncols - 1) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False)

    for ax, (name, df) in zip(axes.flat, pred_data.items()):
        ax.hist(df[df['actual'] == 0]['proba'], bins=30, alpha=0.6,
                color='#e74c3c', density=True, label='actual DOWN')
        ax.hist(df[df['actual'] == 1]['proba'], bins=30, alpha=0.6,
                color='#2ecc71', density=True, label='actual UP')
        ax.axvline(0.5, color='black', lw=0.8, ls='--')
        ax.set_title(name, fontsize=9)
        ax.set_xlabel('P(up)', fontsize=8)
        ax.legend(fontsize=7)

    for ax in axes.flat[len(pred_data):]:
        ax.set_visible(False)

    plt.suptitle('Distribution of predicted P(up) by actual outcome', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

### B2  Calibration  (reliability diagram)

In [ ]:
def calibration_curve_data(probas, actuals, n_bins=10):
    """Return (mean_proba_per_bin, fraction_positive_per_bin, counts)."""
    bins = np.linspace(0, 1, n_bins + 1)
    bin_ids = np.digitize(probas, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)
    mean_p, frac_p, counts = [], [], []
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() > 0:
            mean_p.append(probas[mask].mean())
            frac_p.append(actuals[mask].mean())
            counts.append(mask.sum())
    return np.array(mean_p), np.array(frac_p), np.array(counts)

def ece(probas, actuals, n_bins=10):
    mp, fp, counts = calibration_curve_data(probas, actuals, n_bins)
    return np.sum(counts * np.abs(fp - mp)) / counts.sum()

if not pred_data:
    print('No prediction data.')
else:
    ncols = min(3, len(pred_data))
    nrows = max(1, (len(pred_data) + ncols - 1) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)

    for ax, (name, df) in zip(axes.flat, pred_data.items()):
        probas  = df['proba'].values
        actuals = df['actual'].values
        mp, fp, counts = calibration_curve_data(probas, actuals)
        brier   = np.mean((probas - actuals) ** 2)
        ece_val = ece(probas, actuals)

        ax.plot([0, 1], [0, 1], 'k--', lw=0.9, label='perfect calibration')
        sc = ax.scatter(mp, fp, c=counts, cmap='Blues', s=60 + counts / counts.max() * 80,
                        edgecolors='navy', lw=0.5, zorder=3)
        ax.plot(mp, fp, color='steelblue', lw=1.2)
        plt.colorbar(sc, ax=ax, label='n predictions', fraction=0.04, pad=0.02)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_title(f'{name}\nBrier={brier:.4f}  ECE={ece_val:.4f}', fontsize=9)
        ax.set_xlabel('Mean predicted P(up)', fontsize=8)
        ax.set_ylabel('Fraction actually up', fontsize=8)
        ax.legend(fontsize=7)

    for ax in axes.flat[len(pred_data):]:
        ax.set_visible(False)

    plt.suptitle('Calibration curves — reliability diagrams', fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

### B3  Confidence-Filtered Accuracy
Only act when the model is confident: P(up) > threshold **or** P(up) < 1 − threshold.
We sweep thresholds from 0.50 (all predictions) to 0.75 (very selective).

In [ ]:
THRESHOLDS = np.arange(0.50, 0.76, 0.025)

if not pred_data:
    print('No prediction data.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for name, df in pred_data.items():
        probas  = df['proba'].values
        actuals = df['actual'].values
        accs, coverages = [], []

        for tau in THRESHOLDS:
            keep = (probas >= tau) | (probas <= 1 - tau)
            cov  = keep.mean()
            if keep.sum() == 0:
                accs.append(np.nan)
            else:
                preds_k = (probas[keep] >= 0.5).astype(int)
                from sklearn.metrics import balanced_accuracy_score
                accs.append(balanced_accuracy_score(actuals[keep], preds_k))
            coverages.append(cov)

        axes[0].plot(THRESHOLDS, accs,    marker='o', ms=4, lw=1.5, label=name)
        axes[1].plot(THRESHOLDS, coverages, marker='o', ms=4, lw=1.5, label=name)

    axes[0].axhline(0.5, color='red', ls='--', lw=0.8, label='random')
    axes[0].set_title('Balanced accuracy vs confidence threshold', fontsize=11)
    axes[0].set_xlabel('Confidence threshold τ  (keep if P > τ or P < 1-τ)')
    axes[0].set_ylabel('Balanced accuracy (kept predictions)')
    axes[0].legend(fontsize=8)

    axes[1].set_title('Coverage vs confidence threshold', fontsize=11)
    axes[1].set_xlabel('Confidence threshold τ')
    axes[1].set_ylabel('Fraction of predictions kept')
    axes[1].yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

### B4  Coverage–Precision Tradeoff

In [ ]:
if not pred_data:
    print('No prediction data.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))

    for name, df in pred_data.items():
        probas  = df['proba'].values
        actuals = df['actual'].values
        covs, accs = [], []

        for tau in THRESHOLDS:
            keep = (probas >= tau) | (probas <= 1 - tau)
            if keep.sum() < 10:
                continue
            from sklearn.metrics import balanced_accuracy_score
            preds_k = (probas[keep] >= 0.5).astype(int)
            covs.append(keep.mean())
            accs.append(balanced_accuracy_score(actuals[keep], preds_k))

        ax.plot(covs, accs, marker='o', ms=5, lw=1.5, label=name)
        # annotate tau=0.5 (full coverage) and last point
        if covs:
            ax.annotate('τ=0.5', (covs[0], accs[0]), fontsize=7,
                        xytext=(4, 4), textcoords='offset points')
            ax.annotate(f'τ={THRESHOLDS[len(covs)-1]:.2f}', (covs[-1], accs[-1]),
                        fontsize=7, xytext=(4, -10), textcoords='offset points')

    ax.axhline(0.5, color='red', ls='--', lw=0.8, label='random')
    ax.set_xlabel('Coverage (fraction of predictions kept)', fontsize=10)
    ax.set_ylabel('Balanced accuracy on kept predictions', fontsize=10)
    ax.set_title('Coverage–Accuracy tradeoff', fontsize=12)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

### B5  Summary — Calibration & Confidence Metrics

In [ ]:
from sklearn.metrics import balanced_accuracy_score, roc_auc_score, log_loss

if not pred_data:
    print('No prediction data.')
else:
    CONF_THRESHOLDS = [0.55, 0.60, 0.65]
    sum_rows = []

    for name, df in pred_data.items():
        probas  = df['proba'].values
        actuals = df['actual'].values
        preds   = df['pred'].values

        row = {
            'artifact':    name,
            'n_preds':     len(df),
            'bal_acc@0.50': f'{balanced_accuracy_score(actuals, preds):.4f}',
            'AUC':         f'{roc_auc_score(actuals, probas):.4f}',
            'Brier':       f'{np.mean((probas - actuals)**2):.4f}',
            'ECE':         f'{ece(probas, actuals):.4f}',
        }
        for tau in CONF_THRESHOLDS:
            keep = (probas >= tau) | (probas <= 1 - tau)
            cov  = keep.mean()
            if keep.sum() >= 5:
                p_k = (probas[keep] >= 0.5).astype(int)
                ba  = balanced_accuracy_score(actuals[keep], p_k)
                row[f'bal_acc@{tau:.2f} ({cov:.0%})'] = f'{ba:.4f}'
            else:
                row[f'bal_acc@{tau:.2f}'] = 'too few'
        sum_rows.append(row)

    sum_df = pd.DataFrame(sum_rows).set_index('artifact')
    sum_df.style \
        .set_caption('Calibration & confidence-filtered accuracy summary') \
        .set_table_styles([{'selector': 'caption',
                            'props': 'font-size:13px; font-weight:bold;'}])

---
## Part C — Continuous Model Confidence
For regression artifacts: does prediction magnitude |pred_return| correlate with quality?
- **C1 (no data needed):** CV window IC vs decisiveness (mean |pred_return|).
- **C2–C4 (requires DB):** inference on held-out data, magnitude threshold sweep, summary.


### C1  CV Window: IC vs Decisiveness
`mean_abs_pred` = mean of |predicted log return| across the window — a proxy for how strongly
the model commits to a direction. Does higher commitment correlate with higher IC?


In [ ]:
wdf_c_rows = []
for stem, a in artifacts_cont.items():
    for wi, m in enumerate(a['cv_metrics']):
        wdf_c_rows.append({
            'name':          stem,
            'model':         a['model_key'],
            'window':        wi,
            'ic':            m.get('ic', np.nan),
            'directional_accuracy': m.get('directional_accuracy', np.nan),
            'mean_abs_pred': m.get('mean_abs_pred', np.nan),
        })

wdf_c = pd.DataFrame(wdf_c_rows)

if wdf_c.empty:
    print("No continuous artifacts — Part C skipped.")
else:
    print(wdf_c.head())


In [ ]:
if not wdf_c.empty:
    models_c = wdf_c['name'].unique()
    ncols = min(3, len(models_c))
    nrows = max(1, (len(models_c) + ncols - 1) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), squeeze=False)
    palette = sns.color_palette('tab10', len(models_c))

    for ax, (name, color) in zip(axes.flat, zip(models_c, palette)):
        sub = wdf_c[wdf_c['name'] == name].dropna(subset=['mean_abs_pred', 'ic'])
        if sub.empty:
            ax.set_visible(False); continue
        ax.scatter(sub['mean_abs_pred'], sub['ic'],
                   color=color, alpha=0.75, s=60, edgecolors='white', lw=0.5)
        if len(sub) > 2:
            m_, b_ = np.polyfit(sub['mean_abs_pred'], sub['ic'], 1)
            x_ = np.linspace(sub['mean_abs_pred'].min(), sub['mean_abs_pred'].max(), 50)
            ax.plot(x_, m_ * x_ + b_, color='black', lw=1.2, ls='--')
            r, p = stats.pearsonr(sub['mean_abs_pred'], sub['ic'])
            ax.set_title(f'{name}\nr={r:.2f}, p={p:.3f}', fontsize=9)
        else:
            ax.set_title(name, fontsize=9)
        ax.axhline(0, color='red', lw=0.7, ls=':')
        ax.set_xlabel('Decisiveness (mean |pred return|)', fontsize=8)
        ax.set_ylabel('IC (Spearman)', fontsize=8)

    for ax in axes.flat[len(models_c):]:
        ax.set_visible(False)

    plt.suptitle('Does higher prediction magnitude → better IC?', fontsize=12, y=1.01)
    plt.tight_layout(); plt.show()

    # Correlation summary
    corr_c_rows = []
    for name, sub in wdf_c.groupby('name'):
        sub = sub.dropna(subset=['mean_abs_pred', 'ic'])
        if len(sub) < 3: continue
        r_ic,  p_ic  = stats.pearsonr(sub['mean_abs_pred'], sub['ic'])
        r_da,  p_da  = stats.pearsonr(sub['mean_abs_pred'], sub['directional_accuracy'])
        corr_c_rows.append({
            'artifact': name,
            'model': sub['model'].iloc[0],
            'n_windows': len(sub),
            'r(decisiveness, ic)': f'{r_ic:.3f} (p={p_ic:.3f})',
            'r(decisiveness, dir_acc)': f'{r_da:.3f} (p={p_da:.3f})',
        })
    if corr_c_rows:
        display(pd.DataFrame(corr_c_rows).set_index('artifact'))
    else:
        print("Not enough windows for correlation analysis.")


---
### C2  Inference on held-out data  (continuous)
Loads each continuous final model and runs forward pass on dates after `train_end`.
Returns `(date, ticker, actual_log_ret, pred_log_ret)`. No sigmoid — raw output is the predicted return.

> **Requires SQLite DB access.**


In [ ]:
def run_inference_continuous(artifact: dict, df_micro_raw, df_macro_raw) -> pd.DataFrame:
    """Run continuous final model on dates after train_end.
    Returns DataFrame with columns: date, ticker, actual, pred.
    """
    import torch

    key       = artifact['model_key']
    horizon   = artifact['horizon']
    ft_type   = artifact['ft_type']
    train_end = pd.Timestamp(artifact['train_end'])

    df_macro_arg = df_macro_raw if ft_type == 'macro' else None
    # ml_ready returns y_discrete as 4th element; we need y_cont (5th)
    df, X, y_disc, mask, y_cont = ml_ready(
        horizon, df_micro_raw, df_macro=df_macro_arg, ft_type=ft_type
    )

    dates   = df.loc[mask, 'date'].reset_index(drop=True)
    tickers = df.loc[mask, 'ticker'].reset_index(drop=True)
    X = X.reset_index(drop=True)
    y_cont = y_cont.reset_index(drop=True)

    eval_mask = pd.to_datetime(dates) > train_end
    if eval_mask.sum() == 0:
        print(f'  {key}: no held-out rows after {train_end.date()}')
        return pd.DataFrame()

    if key in ('xgb',):
        X_ev = X.loc[eval_mask]
        preds = artifact['model'].predict(X_ev)
        return pd.DataFrame({
            'date':   dates.loc[eval_mask].values,
            'ticker': tickers.loc[eval_mask].values,
            'actual': y_cont.loc[eval_mask].values,
            'pred':   preds,
        })

    # --- neural (GRU, LSTM, CNN+GRU, CNN+LSTM) ---
    from models.neural.lstm import add_cyclic_dow, build_sequences, SEQ_LEN
    if 'dow' in X.columns:
        X = add_cyclic_dow(X.copy())

    seqs, labs_disc, last_dates = build_sequences(X, y_disc, tickers, dates, SEQ_LEN)
    # Rebuild labs using y_cont for the same sequence indices
    _, labs_cont, _ = build_sequences(X, y_cont, tickers, dates, SEQ_LEN)

    eval_date_set = set(dates.loc[eval_mask].values)
    seq_mask = np.array([d in eval_date_set for d in last_dates])
    seqs_ev  = seqs[seq_mask]
    labs_ev  = labs_cont[seq_mask]
    last_ev  = last_dates[seq_mask]

    if len(seqs_ev) == 0:
        print(f'  {key}: no eval sequences')
        return pd.DataFrame()

    sc = artifact['scaler']
    n, T, f = seqs_ev.shape
    seqs_s = sc.transform(seqs_ev.reshape(-1, f)).reshape(n, T, f)

    cfg = artifact['model_config']
    if 'num_filters' in cfg:
        from models.neural.cnn_rnn import StockCNNRNN
        model = StockCNNRNN(**cfg)
    else:
        from models.neural.lstm import StockRNN
        model = StockRNN(**cfg)
    model.load_state_dict(artifact['model_state'])
    model.eval()

    with torch.no_grad():
        raw_out = model(torch.tensor(seqs_s, dtype=torch.float32))
    # No sigmoid — raw output is predicted log return
    preds = raw_out.numpy().flatten()

    return pd.DataFrame({
        'date':   last_ev,
        'actual': labs_ev,
        'pred':   preds,
    })


# --- collect ---
if not artifacts_cont:
    print("No continuous artifacts — skipping C2-C4.")
    pred_data_cont = {}
else:
    from config import load_env
    load_env()
    # Reuse df_micro_raw / df_macro_raw if already loaded by Part B;
    # otherwise load them here.
    try:
        df_micro_raw
    except NameError:
        df_micro_raw, df_macro_raw = _load_ohlcv()

    pred_data_cont = {}
    for name, a in artifacts_cont.items():
        print(f'Running continuous inference: {name} ...')
        df_pred = run_inference_continuous(a, df_micro_raw, df_macro_raw)
        if len(df_pred) > 0:
            pred_data_cont[name] = df_pred
            print(f'  -> {len(df_pred)} rows, '
                  f'date range: {df_pred["date"].min()} -- {df_pred["date"].max()}')

    print(f'\nContinuous predictions collected for: {list(pred_data_cont.keys())}')


### C3  |pred_return| as Confidence Filter
Act only when |predicted return| ≥ δ. Sweep δ from 0 to the 90th percentile of |pred| in 20 steps.
Shows coverage–directional_accuracy and coverage–IC curves.


In [ ]:
from scipy.stats import spearmanr

if not pred_data_cont:
    print("No continuous prediction data.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for name, df in pred_data_cont.items():
        actuals  = df['actual'].values
        preds    = df['pred'].values
        abs_pred = np.abs(preds)

        p90     = np.percentile(abs_pred, 90)
        deltas  = np.linspace(0, p90, 20)
        covs, dir_accs, ics = [], [], []

        for delta in deltas:
            keep = abs_pred >= delta
            if keep.sum() < 10:
                break
            covs.append(keep.mean())
            dir_accs.append(float(np.mean(np.sign(actuals[keep]) == np.sign(preds[keep]))))
            ic_val, _ = spearmanr(actuals[keep], preds[keep])
            ics.append(float(ic_val) if np.isfinite(ic_val) else 0.0)

        axes[0].plot(covs, dir_accs, marker='o', ms=4, lw=1.5, label=name)
        axes[1].plot(covs, ics,      marker='o', ms=4, lw=1.5, label=name)

    axes[0].axhline(0.5, color='red', ls='--', lw=0.8, label='random')
    axes[0].set_title('Coverage–Directional Accuracy', fontsize=11)
    axes[0].set_xlabel('Coverage (fraction kept)')
    axes[0].set_ylabel('Directional accuracy')
    axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    axes[0].legend(fontsize=8)

    axes[1].axhline(0, color='red', ls='--', lw=0.8, label='IC=0')
    axes[1].set_title('Coverage–IC (Spearman)', fontsize=11)
    axes[1].set_xlabel('Coverage (fraction kept)')
    axes[1].set_ylabel('Spearman IC')
    axes[1].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
    axes[1].legend(fontsize=8)

    plt.suptitle('Does filtering on |pred_return| improve quality?', fontsize=12, y=1.01)
    plt.tight_layout(); plt.show()


### C4  Summary — Continuous Inference Metrics

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error

if not pred_data_cont:
    print("No continuous prediction data.")
else:
    sum_rows_c = []
    for name, df in pred_data_cont.items():
        actuals = df['actual'].values
        preds   = df['pred'].values
        ic_val, _ = spearmanr(actuals, preds)
        sum_rows_c.append({
            'artifact':            name,
            'n_preds':             len(df),
            'MAE':                 f'{mean_absolute_error(actuals, preds):.6f}',
            'RMSE':                f'{np.sqrt(np.mean((actuals - preds)**2)):.6f}',
            'IC (Spearman)':       f'{float(ic_val):.4f}' if np.isfinite(ic_val) else 'nan',
            'Directional Acc':     f'{np.mean(np.sign(actuals) == np.sign(preds)):.4f}',
        })

    pd.DataFrame(sum_rows_c).set_index('artifact').style \
        .set_caption('Continuous inference metrics on held-out data') \
        .set_table_styles([{'selector': 'caption',
                            'props': 'font-size:13px; font-weight:bold;'}])
